Simple Neural Network Model


In [32]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=9, padding=4),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv1d(out_channels, out_channels, kernel_size=9, padding=4),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)
    
def pad_to_match(tensor_to_pad, target_tensor):
    diff = target_tensor.size(2) - tensor_to_pad.size(2)
    if diff > 0:
        pad_left = diff // 2
        pad_right = diff - pad_left
        tensor_to_pad = F.pad(tensor_to_pad, (pad_left, pad_right))
    elif diff < 0:
        # Optional: If tensor_to_pad is longer, crop it
        crop_left = (-diff) // 2
        crop_right = crop_left + target_tensor.size(2)
        tensor_to_pad = tensor_to_pad[:, :, crop_left:crop_right]
    return tensor_to_pad

class UNet1D(nn.Module):
    def __init__(self, in_channels=4, base_channels=32, out_channels=3):
        super().__init__()

        # Encoder
        self.enc1 = ConvBlock1D(in_channels, base_channels)         # [B, 32, L]
        self.pool1 = nn.MaxPool1d(2)                                 # [B, 32, L/2]
        self.enc2 = ConvBlock1D(base_channels, base_channels*2)     # [B, 64, L/2]
        self.pool2 = nn.MaxPool1d(2)                                 # [B, 64, L/4]
        self.enc3 = ConvBlock1D(base_channels*2, base_channels*4)   # [B, 128, L/4]

        # Decoder (upsampling with kernel=2, stride=2 to double length)
        self.up2 = nn.ConvTranspose1d(base_channels*4, base_channels*2, kernel_size=2, stride=2)  # [B, 64, L/2]
        self.dec2 = ConvBlock1D(base_channels*4, base_channels*2)

        self.up1 = nn.ConvTranspose1d(base_channels*2, base_channels, kernel_size=2, stride=2)     # [B, 32, L]
        self.dec1 = ConvBlock1D(base_channels*2, base_channels)

        # Output: map to 3D coordinates per position with padding to keep length
        self.final = nn.Conv1d(base_channels, out_channels, kernel_size=9, padding=4)  # [B, 3, L]

    def forward(self, x):
        e1 = self.enc1(x)                     # [B, 32, L]
        e2 = self.enc2(self.pool1(e1))       # [B, 64, L/2]
        e3 = self.enc3(self.pool2(e2))       # [B, 128, L/4]

        d2 = self.up2(e3)                    # [B, 64, L/2]
        d2 = pad_to_match(d2, e2)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)                   # [B, 32, L]
        d1 = pad_to_match(d1, e1)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        #print(f"d2 shape: {d2.shape}, e2 shape: {e2.shape}")
        #print(f"d1 shape: {d1.shape}, e1 shape: {e1.shape}")

        #return self.final(d1)               # [B, 3, L]
        return self.final(d1).mean(dim=2)  # [B, 3]



Dataset Loader

In [33]:
class RNADataset(torch.utils.data.Dataset):
    def __init__(self, df, window_size=5):
        self.window_size = window_size
        self.data = []
        self.labels = []
        seq = df['resname'].values
        coords = df[['x_1', 'y_1', 'z_1']].values

        mapping = {'A': [1,0,0,0], 'U': [0,1,0,0], 'C': [0,0,1,0], 'G': [0,0,0,1]}
        pad = [0, 0, 0, 0]

        for i in range(len(seq)):
            context = []
            for j in range(i - window_size//2, i + window_size//2 + 1):
                if 0 <= j < len(seq):
                    context.append(mapping[seq[j]])
                else:
                    context.append(pad)
            self.data.append(torch.tensor(context).T)  # shape: [4, window_size]
            self.labels.append(torch.tensor(coords[i]))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx].float(), self.labels[idx].float()


Training the Model

In [34]:
import pandas as pd
import torch
from torch.utils.data import DataLoader

# Load the data
df = pd.read_csv('RNA_data.csv')

# Your RNADataset class (assumed already defined)
class RNADataset(torch.utils.data.Dataset):
    def __init__(self, df, window_size=5):
        self.window_size = window_size
        self.data = []
        self.labels = []

        seq = df['resname'].values
        coords = df[['x_1', 'y_1', 'z_1']].values
        mapping = {'A': [1, 0, 0, 0], 'U': [0, 1, 0, 0], 'C': [0, 0, 1, 0], 'G': [0, 0, 0, 1]}
        pad = [0, 0, 0, 0]

        for i in range(len(seq)):
            context = []
            for j in range(i - window_size//2, i + window_size//2 + 1):
                if 0 <= j < len(seq):
                    context.append(mapping.get(seq[j], pad))
                else:
                    context.append(pad)
            self.data.append(torch.tensor(context).T)  # shape: [4, window_size]
            self.labels.append(torch.tensor(coords[i]))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx].float(), self.labels[idx].float()

# --- Build dataset from multiple molecules ---

all_data = []

for mol_id, group in df.groupby('mol_id'):
    group = group.sort_values('resid').reset_index(drop=True)
    dataset = RNADataset(group, window_size=5)
    all_data.extend([dataset[i] for i in range(len(dataset))])

# Create a DataLoader
loader = DataLoader(all_data, batch_size=32, shuffle=True)

model = UNet1D()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
def tm_score_approx(pred, target, d0=1.24):
    """
    pred, target: (B, N, 3) coordinates
    d0: normalization constant
    """
    dists = torch.norm(pred - target, dim=-1)  # (B, N)
    score = 1 / (1 + (dists / d0) ** 2)
    return score.mean()  # average over residues and batch

for epoch in range(20):
    total_loss = 0
    total_tm = 0
    model.train()

    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)  # (B, N, features), (B, N, 3)

        optimizer.zero_grad()
        outputs = model(inputs)  # (B, N, 3)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        # Surrogate TM-score
        tm = tm_score_approx(outputs, targets)

        total_loss += loss.item()
        total_tm += tm.item()

    avg_loss = total_loss / len(loader)
    avg_tm = total_tm / len(loader)

    print(f"Epoch {epoch+1}: Loss = {avg_loss:.4f}, TM-score ≈ {avg_tm:.4f}")



Epoch 1: Loss = 16776.0079, TM-score ≈ 0.0002
Epoch 2: Loss = 16282.9251, TM-score ≈ 0.0002
Epoch 3: Loss = 16261.0743, TM-score ≈ 0.0002
Epoch 4: Loss = 16239.8829, TM-score ≈ 0.0002
Epoch 5: Loss = 16239.6650, TM-score ≈ 0.0002
Epoch 6: Loss = 16234.4027, TM-score ≈ 0.0002
Epoch 7: Loss = 16223.9166, TM-score ≈ 0.0002
Epoch 8: Loss = 16217.0995, TM-score ≈ 0.0002
Epoch 9: Loss = 16217.1955, TM-score ≈ 0.0002
Epoch 10: Loss = 16210.3790, TM-score ≈ 0.0002
Epoch 11: Loss = 16208.6694, TM-score ≈ 0.0002
Epoch 12: Loss = 16207.8998, TM-score ≈ 0.0002
Epoch 13: Loss = 16200.3199, TM-score ≈ 0.0002
Epoch 14: Loss = 16200.8666, TM-score ≈ 0.0002
Epoch 15: Loss = 16198.8176, TM-score ≈ 0.0002
Epoch 16: Loss = 16192.4197, TM-score ≈ 0.0002
Epoch 17: Loss = 16195.9108, TM-score ≈ 0.0002
Epoch 18: Loss = 16192.9067, TM-score ≈ 0.0002
Epoch 19: Loss = 16185.0877, TM-score ≈ 0.0002
Epoch 20: Loss = 16188.6715, TM-score ≈ 0.0002


In [35]:
# Save the model
torch.save(model.state_dict(), 'RNA_unet.pt')
# Initialize and load the model
model = UNet1D()
model.load_state_dict(torch.load('RNA_unet.pt'))
model.eval()  # Set to evaluation mode


UNet1D(
  (enc1): ConvBlock1D(
    (block): Sequential(
      (0): Conv1d(4, 32, kernel_size=(9,), stride=(1,), padding=(4,))
      (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv1d(32, 32, kernel_size=(9,), stride=(1,), padding=(4,))
      (4): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (pool1): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (enc2): ConvBlock1D(
    (block): Sequential(
      (0): Conv1d(32, 64, kernel_size=(9,), stride=(1,), padding=(4,))
      (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv1d(64, 64, kernel_size=(9,), stride=(1,), padding=(4,))
      (4): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (pool2): MaxPool1d(ke

Evaluate or Visualize
Compare predicted vs. actual 3D coordinates using RMSD, or visualize with matplotlib

In [36]:
import torch
import random
import pandas as pd

def prepare_sequence(seq, window_size=5):
    mapping = {'A': [1, 0, 0, 0], 'U': [0, 1, 0, 0], 'C': [0, 0, 1, 0], 'G': [0, 0, 0, 1]}
    pad = [0, 0, 0, 0]
    input_data = []

    for i in range(len(seq)):
        context = []
        for j in range(i - window_size // 2, i + window_size // 2 + 1):
            if 0 <= j < len(seq):
                base = seq[j]
                one_hot = mapping.get(base, random.choice(list(mapping.values())))
                context.append(one_hot)
            else:
                context.append(pad)
        input_data.append(torch.tensor(context).T.float())  # shape: [4, window_size]

    return input_data

# Example sequence
test_seq = ['A', 'U', 'C', 'G', 'A','G','U','A']
window_size = 5
num_runs = 5

# Store predictions for each run
all_run_predictions = []

for run in range(num_runs):
    inputs = prepare_sequence(test_seq, window_size)
    run_predictions = []

    with torch.no_grad():
        for x in inputs:
            x = x.unsqueeze(0)  # Shape: [1, 4, window_size]
            y_pred = model(x)   # Output: [1, 3]
            run_predictions.append(y_pred.squeeze().numpy())  # [3]

    all_run_predictions.append(run_predictions)  # shape: [seq_len, 3] per run

# Transpose to organize by residue
final_predictions = []
for i, residue in enumerate(test_seq):
    row = [residue, i + 1]  # resname, resid (1-based index)
    for run in range(num_runs):
        row.extend(all_run_predictions[run][i])  # x, y, z for each run
    final_predictions.append(row)

# Define column names
coords_columns = [f'{axis}_{run+1}' for run in range(num_runs) for axis in ['x', 'y', 'z']]
columns = ['resname', 'resid'] + coords_columns

# Save to CSV
df = pd.DataFrame(final_predictions, columns=columns)
df.to_csv("predictions.csv", index=False)


[W NNPACK.cpp:64] Could not initialize NNPACK! Reason: Unsupported hardware.
